<a href="https://colab.research.google.com/github/MukulVerma-ML/upskillCampus/blob/main/Gearbox_Predictive_Maintenance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import os
from scipy.stats import kurtosis, skew

def extract_features(file_path, label, load):
    df = pd.read_csv(file_path, sep=r'\s+', header=None)
    df.columns = ['S1','S2','S3','S4']
    features = {}
    for col in df.columns:
        features[f'{col}_RMS'] = np.sqrt(np.mean(df[col]**2))
        features[f'{col}_Kurtosis'] = kurtosis(df[col])
        features[f'{col}_Skew'] = skew(df[col])
        features[f'{col}_Peak'] = np.max(np.abs(df[col]))
    features['Load'] = load
    features['Label'] = label
    return features

all_data = []

folders = {
    'Healthy Data/healthy': 0, # 0 = Healthy
    'BrokenTooth Data/broken': 1 # 1 = Broken
}

print("Processing files...")
for folder_name, label in folders.items():
    print(f"\nChecking folder: {folder_name}")
    for file in os.listdir(folder_name):
        if file.endswith('.txt'):
            # h30hz10.txt ya b30hz10.txt se load nikalega
            load = int(file.split('hz')[1].replace('.txt',''))
            path = os.path.join(folder_name, file)
            all_data.append(extract_features(path, label, load))
            print(f"Done: {file}")

df_final = pd.DataFrame(all_data)
df_final.to_csv('gearbox_dataset.csv', index=False)

print("\n✅ Ho gaya!")
print("Final Shape:", df_final.shape) # (20, 19) aana chahiye
print("\nLabel count:")
print(df_final['Label'].value_counts()) # 0:10, 1:10
print("\nPreview:")
print(df_final.head())

Processing files...

Checking folder: Healthy Data/healthy
Done: h30hz40.txt
Done: h30hz70.txt
Done: h30hz10.txt
Done: h30hz0.txt
Done: h30hz30.txt
Done: h30hz90.txt
Done: h30hz20.txt
Done: h30hz50.txt
Done: h30hz60.txt
Done: h30hz80.txt

Checking folder: BrokenTooth Data/broken
Done: b30hz80.txt
Done: b30hz90.txt
Done: b30hz60.txt
Done: b30hz40.txt
Done: b30hz70.txt
Done: b30hz30.txt
Done: b30hz10.txt
Done: b30hz20.txt
Done: b30hz0.txt
Done: b30hz50.txt

✅ Ho gaya!
Final Shape: (20, 18)

Label count:
Label
0    10
1    10
Name: count, dtype: int64

Preview:
     S1_RMS  S1_Kurtosis   S1_Skew  S1_Peak    S2_RMS  S2_Kurtosis   S2_Skew  \
0  7.558125     2.659027  0.127076  50.2297  4.508998     2.715138 -0.171858   
1  8.170147     1.145952  0.041974  52.6709  4.617818     1.130256 -0.125848   
2  6.013689     3.403795 -0.084048  43.7555  4.193542     3.644639 -0.181001   
3  5.946125     3.133808 -0.070615  39.2677  4.156309     3.925242 -0.181180   
4  6.961527     2.690139  0.118853 

In [ ]:
#Model training
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. X aur y alag splitting
X = df_final.drop(['Label'], axis=1)
y = df_final['Label']

# 2. Train-Test split. Load ke hisab se karna hai to ye use karna pdega
X_train = X[X['Load'] <= 60]
X_test = X[X['Load'] > 60]
y_train = y[X['Load'] <= 60]
y_test = y[X['Load'] > 60]

# 3. Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Check
pred = model.predict(X_test)
print("\nAccuracy:", accuracy_score(y_test, pred) * 100, "%")
print("\nReport:\n", classification_report(y_test, pred))


Accuracy: 100.0 %

Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         3

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6

